In [3]:
!pip install dbt-core dbt-bigquery -q

from google.colab import auth
auth.authenticate_user()

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/.config/application_default_credentials.json"
print("Xác thực thành công!")

Xác thực thành công!


In [4]:
import os

# Tạo thư mục
os.makedirs("/content/dbt_telco/models/staging", exist_ok=True)

# ----- profiles.yml -----
profiles = """
dbt_telco:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: btqt2-telco-dbt-bigquery
      dataset: silver_telco
      location: asia-southeast1
      threads: 4
      timeout_seconds: 300
      priority: interactive
"""
with open("/content/dbt_telco/profiles.yml", "w") as f:
    f.write(profiles)

# ----- dbt_project.yml -----
dbt_project = """
name: 'dbt_telco'
version: '1.0.0'
config-version: 2

profile: 'dbt_telco'

model-paths: ["models"]
clean-targets:
  - "target"
  - "dbt_packages"

models:
  dbt_telco:
    staging:
      +materialized: view
"""
with open("/content/dbt_telco/dbt_project.yml", "w") as f:
    f.write(dbt_project)

print("Đã tạo cấu trúc project!")

Đã tạo cấu trúc project!


In [5]:
# ----- _sources.yml -----
sources = """
version: 2

sources:
  - name: bronze_telco
    database: btqt2-telco-dbt-bigquery
    schema: bronze_telco
    tables:
      - name: customers_raw
        description: "Raw customer churn data loaded from CSV into Bronze layer"
"""
with open("/content/dbt_telco/models/staging/_sources.yml", "w") as f:
    f.write(sources)

# ----- stg_telco_customers.sql -----
sql = """
with source as (
    select *
    from {{ source('bronze_telco', 'customers_raw') }}
),

renamed_and_casted as (
    select
        trim(customerID)                                    as customer_id,
        trim(gender)                                        as gender,
        cast(SeniorCitizen as int64)                        as senior_citizen,
        trim(Partner)                                       as partner,
        trim(Dependents)                                    as dependents,
        cast(tenure as int64)                               as tenure,
        trim(PhoneService)                                  as phone_service,
        trim(MultipleLines)                                 as multiple_lines,
        trim(InternetService)                               as internet_service,
        trim(OnlineSecurity)                                as online_security,
        trim(OnlineBackup)                                  as online_backup,
        trim(DeviceProtection)                              as device_protection,
        trim(TechSupport)                                   as tech_support,
        trim(StreamingTV)                                   as streaming_tv,
        trim(StreamingMovies)                               as streaming_movies,
        trim(Contract)                                      as contract,
        trim(PaperlessBilling)                              as paperless_billing,
        trim(PaymentMethod)                                 as payment_method,
        safe_cast(MonthlyCharges as numeric)                as monthly_charges,
        safe_cast(
            nullif(trim(TotalCharges), '') as numeric
        )                                                   as total_charges,
        trim(Churn)                                         as churn
    from source
)

select *
from renamed_and_casted
"""
with open("/content/dbt_telco/models/staging/stg_telco_customers.sql", "w") as f:
    f.write(sql)

print("Đã tạo model staging!")

Đã tạo model staging!


In [6]:
!dbt debug --project-dir /content/dbt_telco --profiles-dir /content/dbt_telco
!dbt run --select stg_telco_customers --project-dir /content/dbt_telco --profiles-dir /content/dbt_telco

02:14:08  Running with dbt=1.12.6
02:14:08  dbt version: 1.12.6
02:14:08  python version: 3.13.16
02:14:08  python path: /usr/bin/python3
02:14:08  os info: Linux-6.6.122+-x86_64-with-glibc2.39
02:14:12  Using profiles dir at /content/dbt_telco
02:14:12  Using profiles.yml file at /content/dbt_telco/profiles.yml
02:14:12  Using dbt_project.yml file at /content/dbt_telco/dbt_project.yml
02:14:12  adapter type: bigquery
02:14:12  adapter version: 1.12.1
02:14:13  Configuration:
02:14:13    profiles.yml file [OK found and valid]
02:14:13    dbt_project.yml file [OK found and valid]
02:14:13  Required dependencies:
02:14:13   - git [OK found]

02:14:13  Connection:
02:14:13    method: oauth
02:14:13    database: btqt2-telco-dbt-bigquery
02:14:13    execution_project: btqt2-telco-dbt-bigquery
02:14:13    schema: silver_telco
02:14:13    location: asia-southeast1
02:14:13    priority: interactive
02:14:13    maximum_bytes_billed: None
02:14:13    reservation: None
02:14:13    impersonate_ser

In [7]:
from google.cloud import bigquery
bq = bigquery.Client(project="btqt2-telco-dbt-bigquery", location="asia-southeast1")

# Số dòng
print(bq.query("""
SELECT COUNT(*) AS so_dong
FROM `btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers`
""").to_dataframe())

# 11 dòng tenure = 0
print(bq.query("""
SELECT customer_id, tenure, monthly_charges, total_charges
FROM `btqt2-telco-dbt-bigquery.silver_telco.stg_telco_customers`
WHERE tenure = 0
""").to_dataframe())

   so_dong
0     7043
   customer_id  tenure monthly_charges total_charges
0   2923-ARZLG       0    19.700000000          None
1   7644-OMVMY       0    19.850000000          None
2   2520-SGTTA       0    20.000000000          None
3   3115-CZMZD       0    20.250000000          None
4   3213-VVOLG       0    25.350000000          None
5   4367-NUYAO       0    25.750000000          None
6   4472-LVYGI       0    52.550000000          None
7   1371-DWPAZ       0    56.050000000          None
8   2775-SEFEE       0    61.900000000          None
9   4075-WKNIU       0    73.350000000          None
10  5709-LVOEQ       0    80.850000000          None
